# Phase 5 LSI — partial fine-tuning, every document chunk


In [1]:
%pip install -q -U huggingface_hub tokenizers scikit-learn tqdm


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.3.0 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.3.0 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.3.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.3.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.3.0 w

In [2]:
import time
import os, json, math, random, gc, tempfile, zipfile, uuid, re, hashlib
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfFileSystem
from sklearn.metrics import precision_recall_fscore_support, f1_score, accuracy_score
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
BUCKET_ROOT = 'hf://buckets/A-240031/legal-lm-phase1-bucket'
PHASE1_TAG = 'decoder_126M'
MODEL_PATH = 'checkpoints/decoder_126M/model.pt'
TOKENIZER_PATH = 'tokenizer_126M_decoder/tokenizer.json'
MODEL_CONFIG = {'d_model': 1024, 'n_heads': 8, 'n_kv_heads': 4, 'n_layers': 8, 'd_ff': 2816, 'max_seq_len': 2048}
MAX_SEQ_LEN = 2048
CHUNK_OVERLAP = 64  # number of shared tokens, NOT the stride
CHUNK_ENCODE_BATCH_SIZE = 4  # chunks per backbone call
USE_GRADIENT_CHECKPOINTING = False  # recompute each chunk microbatch during backward
UNFREEZE_LAST_N_BLOCKS = 2  # train these original final blocks, plus final norm and head
TRAIN_MAX_CHUNKS = None
EVAL_MAX_CHUNKS = None
USE_AMP = DEVICE == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16

def autocast_context():
    return torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP)

DATA_PREFIX = 'Phase_5_LSI'
LABEL_KEY = 'labels'
assert 0 < MAX_SEQ_LEN <= MODEL_CONFIG.get('max_seq_len', MODEL_CONFIG.get('max_encoder_len'))
assert 0 <= CHUNK_OVERLAP < MAX_SEQ_LEN
assert CHUNK_ENCODE_BATCH_SIZE > 0
assert 1 <= UNFREEZE_LAST_N_BLOCKS <= MODEL_CONFIG['n_layers']
BATCH_SIZE = 1  # documents per batch; all chunks of each document are used
GRAD_ACCUM_STEPS = 4
LR = 3e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NUM_EPOCHS = 1
PATIENCE = 10
MIN_DELTA = 0.001
ADAPTER_CONFIG = {}  # no adapters
HEAD_DROPOUT = 0.1
UPLOAD_CHECKPOINTS = True
TECHNIQUE = 'partial_fft_allchunks'
TRAIN_FRACTION = 1.0
MAX_TRAIN_DOCS = None  # e.g. 10000; None uses all train documents
LOG_EVERY_DOCS = 10000
MAX_GRAPH_CHUNKS = 16  # replay only larger batches; None disables replay
POS_WEIGHT_MAX = 10.0
PER_LABEL_MIN_SUPPORT = 5
THRESHOLD_GRID = [i / 100 for i in range(5, 81, 5)]

# All loading/saving locations are controlled here.
# Paths below are relative to BUCKET_ROOT; an absolute hf:// path also works.
SAVE_TAG = None  # None generates a descriptive tag; or enter your own run tag.
SUBMISSION_ZIP_NAME = None  # None uses SAVE_TAG + '.zip'; or enter a filename.
CHECKPOINT_PREFIX = 'checkpoints'
RESUME = False  # True resumes SAVE_TAG; set SAVE_TAG to the existing latest tag.
WORK_ROOT = Path('/kaggle/working/lsi_runs') if Path('/kaggle/working').exists() else Path.cwd() / 'lsi_runs'

if RESUME and SAVE_TAG is None:
    raise ValueError('To resume, set SAVE_TAG to the existing run tag.')
if SAVE_TAG is None:
    SAVE_TAG = f'lsi_{PHASE1_TAG}_{TECHNIQUE}_len{MAX_SEQ_LEN}_ov{CHUNK_OVERLAP}_sample{TRAIN_MAX_CHUNKS}_eval{EVAL_MAX_CHUNKS}_blocks{UNFREEZE_LAST_N_BLOCKS}'
if not isinstance(SAVE_TAG, str) or not re.fullmatch(r'[A-Za-z0-9_.-]+', SAVE_TAG) or SAVE_TAG in {'.', '..'}:
    raise ValueError('SAVE_TAG must be a single name containing letters, numbers, _, - or .')
BEST_TAG = SAVE_TAG + '_best'
if SUBMISSION_ZIP_NAME is None:
    SUBMISSION_ZIP_NAME = SAVE_TAG + '.zip'
if Path(SUBMISSION_ZIP_NAME).name != SUBMISSION_ZIP_NAME or not SUBMISSION_ZIP_NAME.endswith('.zip'):
    raise ValueError('SUBMISSION_ZIP_NAME must be a filename ending in .zip.')
WORK_DIR = WORK_ROOT / SAVE_TAG
WORK_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF')
    except Exception:
        pass
fs = HfFileSystem(token=HF_TOKEN)

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
assert LOG_EVERY_DOCS > 0
assert MAX_TRAIN_DOCS is None or (type(MAX_TRAIN_DOCS) is int and MAX_TRAIN_DOCS > 0)
assert MAX_GRAPH_CHUNKS is None or MAX_GRAPH_CHUNKS > 0
print('Device:', DEVICE)
def bucket_path(path):
    if path.startswith('hf://'):
        return path
    return BUCKET_ROOT.rstrip('/') + '/' + path.lstrip('/')

def checkpoint_remote_path(tag):
    return bucket_path(f"{CHECKPOINT_PREFIX.strip('/')}/{tag}/checkpoint.pt")

def download_bucket_file(remote_path):
    # Identity-specific cache directory; refresh the bytes on each call.
    remote = bucket_path(remote_path)
    cache_key = hashlib.sha256(remote.encode()).hexdigest()[:20]
    local = WORK_ROOT / '_inputs' / cache_key / remote.rsplit('/', 1)[-1]
    local.parent.mkdir(parents=True, exist_ok=True)
    temporary = local.with_name(local.name + '.download')
    fs.get_file(remote, str(temporary))
    os.replace(temporary, local)
    return local

def assert_run_destination():
    if RESUME:
        return
    for tag in [SAVE_TAG, BEST_TAG]:
        local = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
        if local.exists() or (UPLOAD_CHECKPOINTS and fs.exists(checkpoint_remote_path(tag))):
            raise FileExistsError(
                f'Checkpoint already exists for {tag}. Choose a new SAVE_TAG, '
                'or set RESUME=True with the original latest SAVE_TAG.'
            )
    manifest = dict(save_tag=SAVE_TAG, best_tag=BEST_TAG, technique=TECHNIQUE,
                    phase1_tag=PHASE1_TAG, model_path=MODEL_PATH,
                    tokenizer_path=TOKENIZER_PATH, submission_zip=SUBMISSION_ZIP_NAME,
                    train_max_chunks=TRAIN_MAX_CHUNKS, eval_max_chunks=EVAL_MAX_CHUNKS,
                    use_amp=USE_AMP, amp_dtype=str(AMP_DTYPE),
                    max_seq_len=MAX_SEQ_LEN, chunk_overlap=CHUNK_OVERLAP, input_format='overlapping_chunks_v1',
                    pooling='token_mean_then_chunk_mean',
                    unfreeze_last_n_blocks=UNFREEZE_LAST_N_BLOCKS, max_train_docs=MAX_TRAIN_DOCS)
    with open(WORK_DIR / 'run_config.json', 'w') as f:
        json.dump(manifest, f, indent=2)

print('Phase-1 model:', bucket_path(MODEL_PATH))
print('Tokenizer:', bucket_path(TOKENIZER_PATH))
print('Latest checkpoint:', checkpoint_remote_path(SAVE_TAG))
print('Best checkpoint:', checkpoint_remote_path(BEST_TAG))
print('Submission ZIP:', WORK_DIR / SUBMISSION_ZIP_NAME)

print('Chunk length/overlap:', MAX_SEQ_LEN, CHUNK_OVERLAP)


Device: cuda
Phase-1 model: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/decoder_126M/model.pt
Tokenizer: hf://buckets/A-240031/legal-lm-phase1-bucket/tokenizer_126M_decoder/tokenizer.json
Latest checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/lsi_decoder_126M_partial_fft_allchunks_len2048_ov64_sampleNone_evalNone_blocks2/checkpoint.pt
Best checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/lsi_decoder_126M_partial_fft_allchunks_len2048_ov64_sampleNone_evalNone_blocks2_best/checkpoint.pt
Submission ZIP: /kaggle/working/lsi_runs/lsi_decoder_126M_partial_fft_allchunks_len2048_ov64_sampleNone_evalNone_blocks2/lsi_decoder_126M_partial_fft_allchunks_len2048_ov64_sampleNone_evalNone_blocks2.zip
Chunk length/overlap: 2048 64


In [3]:
local_tokenizer = download_bucket_file(TOKENIZER_PATH)
tokenizer = Tokenizer.from_file(str(local_tokenizer))
tokenizer.no_padding()
tokenizer.no_truncation()
VOCAB_SIZE = tokenizer.get_vocab_size()
EMPTY_ID = tokenizer.token_to_id('<|eos|>')
PAD_ID = tokenizer.token_to_id('<|pad|>')
if EMPTY_ID is None:
    raise ValueError('Expected the Phase-1 <|eos|> token.')
if PAD_ID is None: PAD_ID = EMPTY_ID


def download_json(filename):
    path = hf_hub_download(repo_id=DATA_REPO, filename=filename, repo_type='dataset', token=HF_TOKEN)
    with open(path, encoding='utf-8') as f: return json.load(f)

def normalize_documents(raw, labelled):
    if isinstance(raw, list): docs = raw
    elif isinstance(raw, dict) and 'id' in raw and 'text' in raw: docs = [raw]
    elif isinstance(raw, dict):
        docs = []
        for doc_id, value in raw.items():
            if not isinstance(value, dict): raise ValueError('Expected ID -> case object mapping')
            doc = dict(value)
            doc.setdefault('id', doc_id)
            if doc['id'] != doc_id: raise ValueError('Document ID mismatch')
            docs.append(doc)
    else: raise ValueError('Expected a list of cases or ID -> case mapping')
    result = []
    for row_index, doc in enumerate(docs):
        case_id, text = doc['id'], doc['text']
        if not isinstance(text, str): raise ValueError(f'{case_id}: text must be a string')
        row = dict(id=case_id, text=text, row_index=row_index)
        if labelled:
            if LABEL_KEY not in doc:
                raise ValueError(f'Missing label field {LABEL_KEY!r}; available fields: {list(doc)}. Set LABEL_KEY.')
            label = doc[LABEL_KEY]
            if not isinstance(label, list) or not label or any(type(x) is not int for x in label):
                raise ValueError(f'{case_id}: expected nonempty list of integer statute IDs, got {label!r}')
            row['labels'] = sorted(set(label))
        result.append(row)
    if not result: raise ValueError('Empty split')
    return result

# Keep each input record separate, even when IDs repeat.
# Test is loaded later, only for prediction.
data = {}
for split in ['train', 'dev']:
    data[split] = normalize_documents(
        download_json(f'{DATA_PREFIX}/Train/{split}.json'), labelled=True)
    counts = Counter(d['id'] for d in data[split])
    print(split, '| records:', len(data[split]), '| unique IDs:', len(counts),
          '| repeated IDs:', sum(n > 1 for n in counts.values()))

# IDs are opaque integers: never assume contiguous IDs or an index offset.
LABELS = sorted({label for split in ('train', 'dev') for d in data[split] for label in d['labels']})
LABEL_TO_INDEX = {label: i for i, label in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
if not NUM_LABELS: raise ValueError('No statute labels')
if set(d['id'] for d in data['train']) & set(d['id'] for d in data['dev']):
    raise ValueError('Train/dev case-ID overlap; resolve before training')
for split in data:
    if len({d['id'] for d in data[split]}) != len(data[split]):
        raise ValueError(f'{split}: duplicate case IDs; resolve explicitly before training')
print('Statutes:', NUM_LABELS, '| IDs:', LABELS)


hf://buckets/A-240031/legal-lm-phase1-bu(…): reconstructing file:   0%|          |  0.00B / 2.47MB            

hf://buckets/A-240031/legal-lm-phase1-bu(…): downloading bytes:           |  0.00B            

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

train | records: 42750 | unique IDs: 42750 | repeated IDs: 0


Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

dev | records: 10181 | unique IDs: 10181 | repeated IDs: 0
Statutes: 100 | IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


In [4]:
# Select documents once, using a reproducible subset. Always retain every chunk.
assert 0 < TRAIN_FRACTION <= 1
original_train_size = len(data['train'])
count = max(1, int(original_train_size * TRAIN_FRACTION))
if MAX_TRAIN_DOCS is not None: count = min(count, MAX_TRAIN_DOCS)
indices = sorted(random.Random(SEED).sample(range(original_train_size), count))
data['train'] = [data['train'][i] for i in indices]
TRAIN_IDS_HASH = hashlib.sha256(json.dumps([d['id'] for d in data['train']], ensure_ascii=False).encode()).hexdigest()
label_counts = Counter(label for d in data['train'] for label in d['labels'])
TRAIN_LABEL_COUNTS = torch.tensor([label_counts[label] for label in LABELS], dtype=torch.float)
POS_WEIGHT = ((len(data['train'])-TRAIN_LABEL_COUNTS)/TRAIN_LABEL_COUNTS.clamp_min(1)).sqrt().clamp(1,POS_WEIGHT_MAX)
POS_WEIGHT[TRAIN_LABEL_COUNTS==0] = 1
print('Training documents:', original_train_size, '->', len(data['train']))
print('Statutes with no positives in selected train documents:', [l for l,c in zip(LABELS,TRAIN_LABEL_COUNTS) if c==0])
print('All chunks are used for each selected document.')


Training documents: 42750 -> 42750
Statutes with no positives in selected train documents: []
All chunks are used for each selected document.


In [5]:
def chunk_token_ids(tokens, max_len=MAX_SEQ_LEN, overlap=CHUNK_OVERLAP):
    if not 0 <= overlap < max_len: raise ValueError('Invalid chunk overlap')
    if not tokens: return [[EMPTY_ID]]
    chunks = []
    for start in range(0, len(tokens), max_len - overlap):
        chunks.append(tokens[start:start + max_len])
        if start + max_len >= len(tokens): break
    return chunks

class LSICaseDataset(Dataset):
    def __init__(self, documents, split):
        self.split, self.epoch = split, 0
        self.rows = []
        token_lengths, chunk_counts = [], []
        for doc in tqdm(documents, desc='Tokenizing cases'):
            tokens = tokenizer.encode(doc['text'], add_special_tokens=False).ids
            chunks = chunk_token_ids(tokens)
            self.rows.append(dict(doc_id=doc['id'], chunks=chunks, labels=doc.get('labels', [])))
            token_lengths.append(len(tokens)); chunk_counts.append(len(chunks))
        lengths = torch.tensor(token_lengths, dtype=torch.float)
        print('Cases/chunks:', len(self.rows), sum(chunk_counts), 'max chunks/case:', max(chunk_counts))
        print('Token lengths p50/p90/p99/max:', torch.quantile(lengths, torch.tensor([.5,.9,.99,1.])).tolist())
    def __len__(self): return len(self.rows)
    def set_epoch(self, epoch): self.epoch = epoch
    def __getitem__(self, index):
        return self.rows[index]

def collate_lsi(rows):
    chunks, owners = [], []
    for case_index, row in enumerate(rows):
        chunks.extend(row['chunks'])
        owners.extend([case_index] * len(row['chunks']))
    order = sorted(range(len(chunks)), key=lambda i: len(chunks[i]), reverse=True)
    chunks = [chunks[i] for i in order]
    owners = [owners[i] for i in order]
    lengths = torch.tensor([len(c) for c in chunks], dtype=torch.long)
    ids = torch.full((len(chunks), int(lengths.max())), PAD_ID, dtype=torch.long)
    for i, chunk in enumerate(chunks): ids[i, :len(chunk)] = torch.tensor(chunk)
    return dict(input_ids=ids, lengths=lengths, chunk_to_case=torch.tensor(owners, dtype=torch.long),
                labels=torch.tensor([[float(label in r['labels']) for label in LABELS] for r in rows], dtype=torch.float),
                doc_ids=[r['doc_id'] for r in rows])

datasets = {split: LSICaseDataset(docs, split) for split, docs in data.items()}
def make_loader(split):
    return DataLoader(datasets[split], batch_size=BATCH_SIZE, shuffle=(split == 'train'),
                      collate_fn=collate_lsi, num_workers=0, pin_memory=(DEVICE == 'cuda'))


Tokenizing cases:   0%|          | 0/42750 [00:00<?, ?it/s]

Cases/chunks: 42750 64711 max chunks/case: 71
Token lengths p50/p90/p99/max: [635.0, 4150.0, 12323.046875, 139406.0]


Tokenizing cases:   0%|          | 0/10181 [00:00<?, ?it/s]

Cases/chunks: 10181 16190 max chunks/case: 59
Token lengths p50/p90/p99/max: [797.0, 4487.0, 13458.435546875, 115496.0]


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2).float() / head_dim)
        )

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, q, k):
        # q, k: [B, H, T, D]
        T = q.size(2)

        cos = self.cos[:T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[:T].unsqueeze(0).unsqueeze(0)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack(
            [q1 * cos - q2 * sin,
             q1 * sin + q2 * cos],
            dim=-1,
        ).flatten(-2)

        k_rot = torch.stack(
            [k1 * cos - k2 * sin,
             k1 * sin + k2 * cos],
            dim=-1,
        ).flatten(-2)

        return q_rot, k_rot


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(
            F.silu(self.gate_proj(x)) * self.up_proj(x)
        )


# ============================================================
# GQA + SDPA
# ============================================================

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len=2048):
        super().__init__()

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

        self.rope = RotaryEmbedding(
            self.head_dim,
            max_seq_len=max_seq_len,
        )

    def forward(self, x):

        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        q, k = self.rope(q, k)

        # Repeat KV heads for GQA
        repeat_factor = self.n_heads // self.n_kv_heads

        k = k.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        # PyTorch SDPA can dispatch to Flash Attention
        # / memory-efficient kernels automatically.
        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
        )

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, -1)
        )

        return self.out_proj(attn_out)


# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        d_ff,
        max_seq_len=2048,
    ):
        super().__init__()

        self.norm1 = nn.RMSNorm(d_model)
        self.attention = GQAAttention(
            d_model,
            n_heads,
            n_kv_heads,
            max_seq_len=max_seq_len,
        )

        self.norm2 = nn.RMSNorm(d_model)
        self.ffn = SwiGLU(
            d_model,
            d_ff,
        )

    def forward(self, x):

        x = x + self.attention(
            self.norm1(x)
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# Decoder-only Language Model
# ============================================================

class EfficientDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=2048,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.token_embedding = nn.Embedding(                             
            vocab_size,
            d_model,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                d_ff=d_ff,
                max_seq_len=max_seq_len,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.RMSNorm(d_model)

        # Tied LM head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.Embedding):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.RMSNorm):
            nn.init.ones_(module.weight)

    def forward(self, input_ids):

        x = self.token_embedding(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        return self.lm_head(x)


In [7]:
from torch.utils.checkpoint import checkpoint as activation_checkpoint

class LSIModel(EfficientDecoderLM):
    def __init__(self, vocab_size, head_dropout=0.1, **kwargs):
        super().__init__(vocab_size=vocab_size, **kwargs)
        del self.lm_head
        self.lsi_dropout = nn.Dropout(head_dropout)
        self.lsi_head = nn.Linear(self.d_model, NUM_LABELS)

    def encode_chunks(self, input_ids, lengths):
        x = self.token_embedding(input_ids)
        # No gradients or intermediate activation storage for the frozen prefix.
        frozen_count = len(self.blocks) - UNFREEZE_LAST_N_BLOCKS
        with torch.no_grad():
            for i in range(frozen_count): x = self.blocks[i](x)
        for i in range(frozen_count, len(self.blocks)): x = self.blocks[i](x)
        x = self.final_norm(x)
        # Right padding cannot affect preceding real tokens under causal attention.
        mask = torch.arange(x.size(1), device=x.device)[None, :] < lengths[:, None]
        return (x.float() * mask.unsqueeze(-1)).sum(1) / lengths[:, None].float()

    def forward(self, input_ids, lengths, chunk_to_case, num_cases):
        device = self.lsi_head.weight.device
        pooled_chunks = []
        for start in range(0, input_ids.size(0), CHUNK_ENCODE_BATCH_SIZE):
            end = min(start + CHUNK_ENCODE_BATCH_SIZE, input_ids.size(0))
            lens = lengths[start:end].to(device, non_blocking=True)
            # Trim this microbatch to its own longest chunk.
            width = int(lengths[start:end].max())
            ids = input_ids[start:end, :width].to(device, non_blocking=True)
            if self.training and USE_GRADIENT_CHECKPOINTING:
                h = activation_checkpoint(self.encode_chunks, ids, lens, use_reentrant=False,
                                          preserve_rng_state=True)
            else: h = self.encode_chunks(ids, lens)
            pooled_chunks.append(h)
        h = torch.cat(pooled_chunks, dim=0)
        owners = chunk_to_case.to(device, non_blocking=True)
        sums = h.new_zeros((num_cases, self.d_model)).index_add(0, owners, h)
        counts = torch.bincount(owners, minlength=num_cases).to(h.dtype).unsqueeze(1)
        assert bool((counts > 0).all()), 'Every case must have at least one chunk'
        document_vectors = sums / counts
        return self.lsi_head(self.lsi_dropout(document_vectors))

def forward_batch(model, batch):
    with autocast_context():
        return model(batch['input_ids'], batch['lengths'], batch['chunk_to_case'], len(batch['doc_ids']))

phase1_path = download_bucket_file(MODEL_PATH)

def build_lsi_model():
    model = LSIModel(VOCAB_SIZE, head_dropout=HEAD_DROPOUT, **MODEL_CONFIG)
    checkpoint = torch.load(phase1_path, map_location='cpu', weights_only=False)
    if 'model_state_dict' in checkpoint:
        state = checkpoint['model_state_dict']
    elif 'model' in checkpoint:
        state = checkpoint['model']
    elif 'state_dict' in checkpoint:
        state = checkpoint['state_dict']
    elif all(isinstance(v, torch.Tensor) for v in checkpoint.values()):
        state = checkpoint
    else:
        raise ValueError(f'Cannot identify model weights. Checkpoint keys: {list(checkpoint)}')
    state = dict(state)
    state.pop('lm_head.weight', None)
    missing, unexpected = model.load_state_dict(state, strict=False)
    assert set(missing) == {'lsi_head.weight', 'lsi_head.bias'} and not unexpected, (missing, unexpected)
    del state, checkpoint
    for p in model.parameters(): p.requires_grad = False
    for block in model.blocks[-UNFREEZE_LAST_N_BLOCKS:]:
        for p in block.parameters(): p.requires_grad = True
    for module in (model.final_norm, model.lsi_head):
        for p in module.parameters(): p.requires_grad = True
    model = model.to(DEVICE)
    print('Total/trainable parameters:', sum(p.numel() for p in model.parameters()),
          sum(p.numel() for p in model.parameters() if p.requires_grad))
    return model

def backward_document_batch(model, batch, scaler):
    y = batch['labels'].to(DEVICE, non_blocking=True)
    n = len(y)
    if MAX_GRAPH_CHUNKS is None or len(batch['lengths']) <= MAX_GRAPH_CHUNKS:
        logits = forward_batch(model, batch)
        loss_sum = F.binary_cross_entropy_with_logits(logits.float(), y,
            pos_weight=POS_WEIGHT.to(DEVICE), reduction='none').mean(-1).sum()
        scaler.scale(loss_sum).backward()
        return loss_sum.detach().item()

    # The original backbone has no dropout or running-state layers.
    # The head dropout is evaluated only once, when its loss is computed below.
    if any(isinstance(m, nn.Dropout) and m.p > 0 for block in model.blocks for m in block.modules()):
        raise RuntimeError('Chunk replay requires deterministic backbone computation; disable replay for a stochastic backbone.')
    owners = batch['chunk_to_case'].to(DEVICE)
    counts = torch.bincount(owners, minlength=n).float().unsqueeze(1)
    sums = torch.zeros((n, model.d_model), device=DEVICE, dtype=torch.float32)
    with torch.no_grad(), autocast_context():
        for start in range(0, len(owners), CHUNK_ENCODE_BATCH_SIZE):
            end = min(start+CHUNK_ENCODE_BATCH_SIZE, len(owners))
            lengths = batch['lengths'][start:end].to(DEVICE)
            width = int(batch['lengths'][start:end].max())
            ids = batch['input_ids'][start:end, :width].to(DEVICE)
            h = model.encode_chunks(ids, lengths)
            sums.index_add_(0, owners[start:end], h.float())
    vectors = (sums/counts).detach().requires_grad_(True)
    with autocast_context():
        logits = model.lsi_head(model.lsi_dropout(vectors))
        loss_sum = F.binary_cross_entropy_with_logits(logits.float(), y,
            pos_weight=POS_WEIGHT.to(DEVICE), reduction='none').mean(-1).sum()
    scaler.scale(loss_sum).backward()
    # Already scaled by GradScaler. Do not apply scaler.scale a second time.
    chunk_gradient = vectors.grad.detach()/counts
    for start in range(0, len(owners), CHUNK_ENCODE_BATCH_SIZE):
        end = min(start+CHUNK_ENCODE_BATCH_SIZE, len(owners))
        lengths = batch['lengths'][start:end].to(DEVICE)
        width = int(batch['lengths'][start:end].max())
        ids = batch['input_ids'][start:end, :width].to(DEVICE)
        with autocast_context(): h = model.encode_chunks(ids, lengths)
        surrogate = (h.float()*chunk_gradient[owners[start:end]]).sum()
        surrogate.backward()
    return loss_sum.detach().item()


hf://buckets/A-240031/legal-lm-phase1-bu(…): reconstructing file:   0%|          |  0.00B /  515MB            

hf://buckets/A-240031/legal-lm-phase1-bu(…): downloading bytes:           |  0.00B            

In [8]:
def experiment_metadata(domain='LSI'):
    return dict(domain=domain, phase1_tag=PHASE1_TAG, model_path=MODEL_PATH, tokenizer_path=TOKENIZER_PATH,
                labels=LABELS, model_config=MODEL_CONFIG, adapter_config=ADAPTER_CONFIG,
                max_seq_len=MAX_SEQ_LEN, chunk_overlap=CHUNK_OVERLAP,
                chunk_encode_batch_size=CHUNK_ENCODE_BATCH_SIZE,
                gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
                train_max_chunks=TRAIN_MAX_CHUNKS, eval_max_chunks=EVAL_MAX_CHUNKS,
                sampling='all_chunks', use_amp=USE_AMP, amp_dtype=str(AMP_DTYPE),
                unfreeze_last_n_blocks=UNFREEZE_LAST_N_BLOCKS,
                pooling='token_mean_then_chunk_mean', input_format='overlapping_chunks_v1',
                data_repo=DATA_REPO, data_prefix=DATA_PREFIX, label_key=LABEL_KEY,
                technique=TECHNIQUE, head_dropout=HEAD_DROPOUT,
                train_fraction=TRAIN_FRACTION, max_train_docs=MAX_TRAIN_DOCS, train_ids_hash=TRAIN_IDS_HASH,
                max_graph_chunks=MAX_GRAPH_CHUNKS, pos_weight=POS_WEIGHT.tolist(), threshold_grid=THRESHOLD_GRID,
                per_label_min_support=PER_LABEL_MIN_SUPPORT, seed=SEED, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
                lr=LR, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
                patience=PATIENCE, min_delta=MIN_DELTA)

def save_lsi_checkpoint(tag, model, optimizer, epoch, extra, domain):
    folder = WORK_DIR / 'checkpoints' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rng = {'python': random.getstate(), 'torch': torch.get_rng_state()}
    if torch.cuda.is_available(): rng['cuda'] = torch.cuda.get_rng_state_all()
    payload = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), rng=rng,
                   epoch=epoch, extra=extra, metadata=experiment_metadata(domain))
    tmp = folder / 'checkpoint.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, folder / 'checkpoint.pt')
    if UPLOAD_CHECKPOINTS:
        fs.put_file(str(folder / 'checkpoint.pt'), checkpoint_remote_path(tag))

def load_lsi_checkpoint(tag, model, domain, optimizer=None, restore_rng=False):
    path = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
    if not path.exists() or restore_rng:
        if UPLOAD_CHECKPOINTS:
            path = download_bucket_file(checkpoint_remote_path(tag))
        elif not path.exists():
            raise FileNotFoundError(path)
    payload = torch.load(path, map_location='cpu', weights_only=False)  # own trusted checkpoints only
    assert payload['metadata'] == experiment_metadata(domain), 'Checkpoint configuration mismatch'
    model.load_state_dict(payload['model'])
    if optimizer is not None: optimizer.load_state_dict(payload['optimizer'])
    if restore_rng:
        random.setstate(payload['rng']['python'])
        torch.set_rng_state(payload['rng']['torch'])
        if torch.cuda.is_available() and 'cuda' in payload['rng']:
            torch.cuda.set_rng_state_all(payload['rng']['cuda'])
    return payload

@torch.no_grad()
def evaluate_lsi(model, loader, thresholds=None, tune=True):
    model.eval()
    targets, scores = [], []
    total_loss = total_n = 0
    for batch in loader:
        y = batch['labels'].to(DEVICE)
        logits = forward_batch(model, batch).float()
        total_loss += F.binary_cross_entropy_with_logits(logits, y, pos_weight=POS_WEIGHT.to(DEVICE), reduction='none').mean(-1).sum().item()
        total_n += len(y)
        targets.append(y.cpu()); scores.append(logits.sigmoid().cpu())
    y = torch.cat(targets).numpy().astype('int64')
    p = torch.cat(scores).numpy()
    # Select on dev only, with the same >= comparison used during test prediction.
    if tune:
        global_t = max(THRESHOLD_GRID, key=lambda t: (f1_score(y, p >= t, average='macro', zero_division=0), -abs(t-.5)))
        thresholds = [global_t] * NUM_LABELS
        for j in range(NUM_LABELS):
            support = int(y[:, j].sum())
            if PER_LABEL_MIN_SUPPORT <= support < len(y):
                candidates = sorted(set(THRESHOLD_GRID + [global_t]))
                thresholds[j] = max(candidates, key=lambda t: (f1_score(y[:,j], p[:,j]>=t, zero_division=0), -abs(t-global_t)))
    if thresholds is None: thresholds = [.5] * NUM_LABELS
    pred = p >= thresholds
    precision, recall, f1, support = precision_recall_fscore_support(y, pred, average=None, zero_division=0)
    return dict(loss=total_loss/total_n, macro_f1=float(f1.mean()),
                micro_f1=float(f1_score(y,pred,average='micro',zero_division=0)),
                subset_accuracy=float(accuracy_score(y,pred)),
                macro_precision=float(precision.mean()), macro_recall=float(recall.mean()),
                thresholds=[float(t) for t in thresholds],
                per_label={int(label):dict(precision=float(precision[j]),recall=float(recall[j]),f1=float(f1[j]),support=int(support[j])) for j,label in enumerate(LABELS)})
def cleanup_latest_checkpoint():
    best_local = WORK_DIR / 'checkpoints' / BEST_TAG / 'checkpoint.pt'
    if UPLOAD_CHECKPOINTS:
        best_remote = checkpoint_remote_path(BEST_TAG)
        if not fs.exists(best_remote):
            print('Best checkpoint missing in bucket; keeping latest.'); return
        if not best_local.is_file():
            best_local = download_bucket_file(best_remote)
        if fs.info(best_remote)['size'] != best_local.stat().st_size:
            print('Best size mismatch; keeping latest.'); return
        latest_remote = checkpoint_remote_path(SAVE_TAG)
        assert latest_remote != best_remote and latest_remote != bucket_path(MODEL_PATH)
        if fs.exists(latest_remote): fs.rm(latest_remote)
    elif not best_local.is_file():
        print('Best checkpoint missing locally; keeping latest.'); return
    latest_local = WORK_DIR / 'checkpoints' / SAVE_TAG / 'checkpoint.pt'
    if latest_local.is_file(): latest_local.unlink()
    print('Retained best checkpoint only:', BEST_TAG)


In [9]:
def train_lsi(domain='LSI'):
    assert domain == 'LSI'
    assert_run_destination()
    seed_everything(SEED)
    model = build_lsi_model()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16)
    tag = SAVE_TAG
    best_tag = BEST_TAG
    start = 0
    extra = dict(best_f1=-1.0, patience_reference=-1.0, stale=0, history=[], best_tag=best_tag)
    if RESUME:
        checkpoint = load_lsi_checkpoint(SAVE_TAG, model, domain, optimizer, restore_rng=True)
        start, extra = checkpoint['epoch'], checkpoint['extra']
        best_tag = extra['best_tag']
        if 'scaler' in extra: scaler.load_state_dict(extra['scaler'])
        del checkpoint
    train_loader, dev_loader = make_loader('train'), make_loader('dev')
    assert len(train_loader) > 0
    for epoch in range(start, NUM_EPOCHS):
        if extra['stale'] >= PATIENCE: break
        datasets['train'].set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_sum = train_n = group_n = 0
        window_sum = window_n = chunks_seen = tokens_seen = replay_batches = 0
        next_log = LOG_EVERY_DOCS
        epoch_started = time.perf_counter()
        for step, batch in enumerate(tqdm(train_loader, desc=f'{domain} epoch {epoch+1}')):
            loss_value = backward_document_batch(model, batch, scaler)
            batch_n = len(batch['doc_ids'])
            group_n += batch_n
            train_n += batch_n
            train_sum += loss_value
            window_n += batch_n
            window_sum += loss_value
            chunks_seen += len(batch['lengths'])
            tokens_seen += int(batch['lengths'].sum())
            replay_batches += int(MAX_GRAPH_CHUNKS is not None and len(batch['lengths']) > MAX_GRAPH_CHUNKS)
            if train_n >= next_log or step + 1 == len(train_loader):
                elapsed = time.perf_counter() - epoch_started
                print(f"Epoch {epoch+1} | docs {train_n:,}/{len(datasets['train']):,} | "
                      f"window loss {window_sum/window_n:.5f} | epoch loss {train_sum/train_n:.5f} | "
                      f"chunks {chunks_seen:,} | tokens/s {tokens_seen/max(elapsed,1e-6):,.0f} | "
                      f"replay batches {replay_batches}", flush=True)
                window_sum = window_n = 0
                while next_log <= train_n: next_log += LOG_EVERY_DOCS
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == len(train_loader):
                scaler.unscale_(optimizer)
                for p in params:
                    if p.grad is not None: p.grad.div_(group_n)
                torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                group_n = 0
        train_seconds = time.perf_counter()-epoch_started
        metrics = evaluate_lsi(model, dev_loader)
        f1 = metrics['macro_f1']
        improved = f1 > extra['best_f1']
        if improved:
            extra['best_f1'] = f1
            extra['thresholds'] = metrics['thresholds']
        if f1 > extra['patience_reference'] + MIN_DELTA:
            extra['patience_reference'], extra['stale'] = f1, 0
        else: extra['stale'] += 1
        extra['history'].append(dict(epoch=epoch+1, train_loss=train_sum/train_n,
            train_documents=train_n, train_chunks=chunks_seen, train_tokens=tokens_seen,
            train_seconds=train_seconds, replay_batches=replay_batches, **metrics))
        print(f"LSI train loss={train_sum/train_n:.4f}, dev macro-F1={f1:.4f}, precision={metrics['macro_precision']:.4f}, recall={metrics['macro_recall']:.4f}, patience={extra['stale']}/{PATIENCE}")
        extra['scaler'] = scaler.state_dict()
        if improved: save_lsi_checkpoint(best_tag, model, optimizer, epoch+1, extra, domain)
        save_lsi_checkpoint(tag, model, optimizer, epoch+1, extra, domain)
    best_payload = load_lsi_checkpoint(best_tag, model, domain)
    model.thresholds = best_payload['extra']['thresholds']
    del best_payload
    metrics = evaluate_lsi(model, dev_loader, thresholds=model.thresholds, tune=False)
    with open(WORK_DIR / f'{domain}_dev_metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
    with open(WORK_DIR / f'{domain}_history.json', 'w') as f: json.dump(extra['history'], f, indent=2)
    print('Best development metrics:', metrics)
    with open(WORK_DIR / 'label_thresholds.json', 'w') as f:
        json.dump(dict(labels=LABELS, thresholds=model.thresholds), f, indent=2)
    cleanup_latest_checkpoint()
    del optimizer
    return model, extra['history']


In [10]:
def validate_predictions(predictions, documents):
    assert len(predictions) == len(documents)
    assert [p['id'] for p in predictions] == [d['id'] for d in documents]
    assert len({p['id'] for p in predictions}) == len(predictions)
    assert all(set(p) == {'id','pred'} and isinstance(p['pred'],list) and len(p['pred']) == len(set(p['pred'])) and all(type(x) is int and x in LABEL_TO_INDEX for x in p['pred']) for p in predictions)

@torch.no_grad()
def predict_lsi(model):
    model.eval()
    predictions = []
    for batch in tqdm(make_loader('test'), desc='LSI test predictions'):
        scores = forward_batch(model, batch).float().sigmoid().cpu()
        selected = scores >= torch.tensor(model.thresholds)[None, :]
        predictions.extend(dict(id=doc_id, pred=[int(LABELS[j]) for j in row.nonzero().flatten().tolist()]) for doc_id,row in zip(batch['doc_ids'],selected))
    validate_predictions(predictions, data['test'])
    return predictions

lsi_model, lsi_history = train_lsi()
# Load test only after training. Its IDs are unique in the supplied data.
data['test'] = normalize_documents(
    download_json(f'{DATA_PREFIX}/Test/test.json'), labelled=False)
datasets['test'] = LSICaseDataset(data['test'], 'test')
predictions = predict_lsi(lsi_model)
json_path = WORK_DIR / 'prediction.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
del lsi_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


Total/trainable parameters: 128761956 23700580


LSI epoch 1:   0%|          | 0/42750 [00:00<?, ?it/s]

Epoch 1 | docs 10,000/42,750 | window loss 0.40423 | epoch loss 0.40423 | chunks 15,194 | tokens/s 4,708 | replay batches 15
Epoch 1 | docs 20,000/42,750 | window loss 0.33737 | epoch loss 0.37080 | chunks 30,179 | tokens/s 4,730 | replay batches 31
Epoch 1 | docs 30,000/42,750 | window loss 0.31622 | epoch loss 0.35261 | chunks 45,212 | tokens/s 4,729 | replay batches 46
Epoch 1 | docs 40,000/42,750 | window loss 0.31059 | epoch loss 0.34210 | chunks 60,408 | tokens/s 4,733 | replay batches 62
Epoch 1 | docs 42,750/42,750 | window loss 0.29895 | epoch loss 0.33933 | chunks 64,711 | tokens/s 4,722 | replay batches 73
LSI train loss=0.3393, dev macro-F1=0.3211, precision=0.2916, recall=0.4026, patience=0/10


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

... (5 outputs left out)

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Tokenizing cases:   0%|          | 0/13019 [00:00<?, ?it/s]

Cases/chunks: 13019 20354 max chunks/case: 112
Token lengths p50/p90/p99/max: [786.0, 4267.0, 13114.5, 220937.0]


LSI test predictions:   0%|          | 0/13019 [00:00<?, ?it/s]

In [11]:
# Packaging can be rerun without retraining.
json_path = WORK_DIR / 'prediction.json'
with open(json_path, encoding='utf-8') as f: predictions = json.load(f)
validate_predictions(predictions, data['test'])
zip_path = WORK_DIR / SUBMISSION_ZIP_NAME
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname='prediction.json')
print('JSON:', json_path)
print('ZIP:', zip_path)
print('Cases:', len(predictions))


JSON: /kaggle/working/lsi_runs/lsi_decoder_126M_partial_fft_allchunks_len2048_ov64_sampleNone_evalNone_blocks2/prediction.json
ZIP: /kaggle/working/lsi_runs/lsi_decoder_126M_partial_fft_allchunks_len2048_ov64_sampleNone_evalNone_blocks2/lsi_decoder_126M_partial_fft_allchunks_len2048_ov64_sampleNone_evalNone_blocks2.zip
Cases: 13019
